## 02_osm_processing
  Splits each country OSM extract into bounding-box tiles (osmium extract) and reads four
  layers per tile (driving network, walking network, buildings, POIs) with pyrosm.
  Input: data/raw/osm/. Output: per-tile parquet layers in data/interim/osm/. Environment: nuts3-scaling.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyrosm

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "osm"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

NUTS3_PATH = RAW_DIR / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
OSM_DIR = RAW_DIR / "osm"

COUNTRY_CODES = ["hu", "pl", "cz", "sk"]
CNTR_CODE_MAP = {"hu": "HU", "pl": "PL", "cz": "CZ", "sk": "SK"}

nuts3_gdf = gpd.read_file(NUTS3_PATH)
v4_nuts3 = nuts3_gdf[nuts3_gdf["CNTR_CODE"].isin(CNTR_CODE_MAP.values())].copy()

print(f"V4 NUTS3 régiók betöltve: {len(v4_nuts3)}")
print(v4_nuts3.groupby("CNTR_CODE").size())

In [ ]:
BUFFER_DEG = 0.01  # kb. 1 km puffer a határon átnyúló utak/POI-k elvesztésének csökkentésére

def region_bbox(geom, buffer_deg: float = BUFFER_DEG) -> list[float]:
    minx, miny, maxx, maxy = geom.bounds
    return [minx - buffer_deg, miny - buffer_deg, maxx + buffer_deg, maxy + buffer_deg]

v4_nuts3["bbox"] = v4_nuts3["geometry"].apply(region_bbox)
v4_nuts3["country_code"] = v4_nuts3["CNTR_CODE"].str.lower()

# gyors ellenőrzés: bbox-szélesség/magasság eloszlása, hogy lássuk, van-e kiugróan nagy régió
v4_nuts3["bbox_width_deg"] = v4_nuts3["bbox"].apply(lambda b: b[2] - b[0])
v4_nuts3["bbox_height_deg"] = v4_nuts3["bbox"].apply(lambda b: b[3] - b[1])

print(v4_nuts3[["NUTS_ID", "NAME_LATN", "country_code", "bbox_width_deg", "bbox_height_deg"]]
      .sort_values("bbox_width_deg", ascending=False)
      .head(5))

In [ ]:
TILE_GRID = {
    "hu": (2, 2),
    "cz": (2, 2),
    "sk": (2, 2),
    "pl": (3, 3),  # PL jelentősen nagyobb terület és fájlméret (2,0 GB), finomabb rács kell
}

def country_tiles(country_geoms, n_x: int, n_y: int) -> list[list[float]]:
    minx, miny, maxx, maxy = country_geoms.total_bounds
    xs = [minx + i * (maxx - minx) / n_x for i in range(n_x + 1)]
    ys = [miny + j * (maxy - miny) / n_y for j in range(n_y + 1)]
    tiles = []
    for i in range(n_x):
        for j in range(n_y):
            tiles.append([xs[i], ys[j], xs[i + 1], ys[j + 1]])
    return tiles

country_tile_map = {}
for code in COUNTRY_CODES:
    country_geoms = v4_nuts3[v4_nuts3["country_code"] == code]
    n_x, n_y = TILE_GRID[code]
    country_tile_map[code] = country_tiles(country_geoms, n_x, n_y)
    print(f"{code}: {len(country_tile_map[code])} csempe ({n_x}x{n_y})")

In [ ]:
import json

EXTRACT_DIR = PROJECT_ROOT / "data" / "interim" / "osm_tiles"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = {"hu": 4, "cz": 4, "sk": 4, "pl": 3}  # PL: 9 tile -> 3 köteg, 3 tile/köteg

def build_batch_configs(country_code: str, tiles: list[list[float]], batch_size: int) -> list[Path]:
    config_paths_batch = []
    for batch_idx in range(0, len(tiles), batch_size):
        batch_tiles = tiles[batch_idx:batch_idx + batch_size]
        extracts = [
            {
                "output": f"{country_code}_tile{batch_idx + i:02d}.osm.pbf",
                "output_format": "pbf",
                "bbox": bbox,
            }
            for i, bbox in enumerate(batch_tiles)
        ]
        config = {"directory": str(EXTRACT_DIR), "extracts": extracts}
        config_path = EXTRACT_DIR / f"{country_code}_extract_config_batch{batch_idx // batch_size:02d}.json"
        config_path.write_text(json.dumps(config, indent=2))
        config_paths_batch.append(config_path)
    return config_paths_batch

country_batch_configs = {}
for code in COUNTRY_CODES:
    paths = build_batch_configs(code, country_tile_map[code], BATCH_SIZE[code])
    country_batch_configs[code] = paths
    print(f"{code}: {len(paths)} köteg")

In [ ]:
import subprocess
import time

def run_osmium_extract(config_path: Path, pbf_path: Path) -> int:
    cmd = [
        "osmium", "extract",
        "--config", str(config_path),
        "--strategy", "complete_ways",
        "--overwrite",
        "--progress",
        str(pbf_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  STDERR: {result.stderr[-1000:]}")
    return result.returncode

for code in COUNTRY_CODES:
    pbf_path = OSM_DIR / f"{code}-latest.osm.pbf"
    for i, config_path in enumerate(country_batch_configs[code]):
        t0 = time.time()
        rc = run_osmium_extract(config_path, pbf_path)
        print(f"{code} köteg {i+1}/{len(country_batch_configs[code])}: {time.time()-t0:.1f} mp, returncode={rc}")

In [ ]:
tile_files = sorted(EXTRACT_DIR.glob("*_tile*.osm.pbf"))
print(f"Kivágott csempe-fájlok száma: {len(tile_files)} (várt: {sum(len(t) for t in country_tile_map.values())})")

for f in tile_files:
    size = f.stat().st_size
    status = "OK " if size > 1000 else "GYANÚS (túl kicsi)"
    print(f"  [{status}] {f.relative_to(PROJECT_ROOT)} ({size:,} byte)")

In [ ]:
import time

def load_layer_from_tiles(tile_paths: list[Path], layer_type: str = "driving") -> gpd.GeoDataFrame:
    chunks = []
    for tp in tile_paths:
        t0 = time.time()
        osm = pyrosm.OSM(str(tp))
        gdf = osm.get_network(network_type=layer_type)
        elapsed = time.time() - t0
        n = len(gdf) if gdf is not None else 0
        print(f"  {tp.name}: {n} elem, {elapsed:.1f} mp")
        if gdf is not None and len(gdf) > 0:
            chunks.append(gdf)
    if not chunks:
        return gpd.GeoDataFrame()
    combined = pd.concat(chunks, ignore_index=True)
    if "id" in combined.columns:
        combined = combined.drop_duplicates(subset="id")
    return combined

hu_tiles = sorted(EXTRACT_DIR.glob("hu_tile*.osm.pbf"))
t_start = time.time()
hu_driving = load_layer_from_tiles(hu_tiles, "driving")
print(f"\nHU driving összesen: {len(hu_driving)} elem, teljes idő: {time.time() - t_start:.1f} mp")

In [ ]:
import re
import time
import json as jsonlib

def sanitize_for_parquet(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    for col in gdf.columns:
        if col == "geometry":
            continue
        if gdf[col].apply(lambda v: isinstance(v, (dict, list))).any():
            gdf[col] = gdf[col].apply(lambda v: jsonlib.dumps(v) if isinstance(v, (dict, list)) else v)
        if gdf[col].dtype == "object":
            gdf[col] = gdf[col].astype(str).replace("None", None)
    return gdf

INTERIM_OSM_DIR = PROJECT_ROOT / "data" / "interim" / "osm"
INTERIM_OSM_DIR.mkdir(parents=True, exist_ok=True)

GYANUS_PL_TILES = [3, 4, 6, 7]

def save_tile_layer(country_code: str, tile_id: str, layer_name: str, gdf) -> None:
    if gdf is None or len(gdf) == 0:
        return
    out_path = INTERIM_OSM_DIR / f"{country_code}_{layer_name}_tile{tile_id}.parquet"
    if out_path.exists():
        print(f"    [skip] {out_path.relative_to(PROJECT_ROOT)} már megvan")
        return
    gdf_clean = sanitize_for_parquet(gdf)
    gdf_clean.to_parquet(out_path)
    print(f"    mentve: {out_path.relative_to(PROJECT_ROOT)} ({len(gdf)} elem)")

def process_single_tile(country_code: str, tile_id: str, tile_path: Path) -> None:
    expected = [
        INTERIM_OSM_DIR / f"{country_code}_network_driving_tile{tile_id}.parquet",
        INTERIM_OSM_DIR / f"{country_code}_network_walking_tile{tile_id}.parquet",
        INTERIM_OSM_DIR / f"{country_code}_buildings_tile{tile_id}.parquet",
        INTERIM_OSM_DIR / f"{country_code}_pois_tile{tile_id}.parquet",
    ]
    if all(f.exists() for f in expected):
        print(f"  {tile_path.name}: [skip] mind a 4 réteg már megvan")
        return

    t0 = time.time()
    osm = pyrosm.OSM(str(tile_path))

    driving = osm.get_network(network_type="driving")
    save_tile_layer(country_code, tile_id, "network_driving", driving)
    del driving

    walking = osm.get_network(network_type="walking")
    save_tile_layer(country_code, tile_id, "network_walking", walking)
    del walking

    buildings = osm.get_buildings()
    save_tile_layer(country_code, tile_id, "buildings", buildings)
    del buildings

    pois = osm.get_pois()
    save_tile_layer(country_code, tile_id, "pois", pois)
    del pois

    del osm
    print(f"  {tile_path.name}: kész ({time.time()-t0:.1f} mp)")

def build_tile_manifest(country_code: str) -> list[tuple[str, Path]]:
    """(tile_id, tile_path) párok — a gyanús PL-csempéket subtile-jaikkal helyettesíti."""
    manifest = []
    tiles = sorted(EXTRACT_DIR.glob(f"{country_code}_tile[0-9][0-9].osm.pbf"))
    for tp in tiles:
        match = re.match(rf"{country_code}_tile(\d{{2}})$", tp.name.removesuffix(".osm.pbf"))
        idx = int(match.group(1))
        if country_code == "pl" and idx in GYANUS_PL_TILES:
            subtiles = sorted(EXTRACT_DIR.glob(f"pl_tile{idx:02d}subtile*.osm.pbf"))
            for stp in subtiles:
                sub_id = stp.name.removesuffix(".osm.pbf").replace(f"{country_code}_tile", "")
                manifest.append((sub_id, stp))
        else:
            manifest.append((f"{idx:02d}", tp))
    return manifest

def process_country_tiles_streaming(country_code: str) -> None:
    manifest = build_tile_manifest(country_code)
    for tile_id, tile_path in manifest:
        process_single_tile(country_code, tile_id, tile_path)

In [ ]:
def subdivide_bbox(bbox: list[float], n_x: int, n_y: int) -> list[list[float]]:
    minx, miny, maxx, maxy = bbox
    xs = [minx + i * (maxx - minx) / n_x for i in range(n_x + 1)]
    ys = [miny + j * (maxy - miny) / n_y for j in range(n_y + 1)]
    return [[xs[i], ys[j], xs[i+1], ys[j+1]] for i in range(n_x) for j in range(n_y)]

subtile_extracts = []
for idx in GYANUS_PL_TILES:
    bbox = country_tile_map["pl"][idx]
    for j, sub_bbox in enumerate(subdivide_bbox(bbox, 2, 2)):
        subtile_extracts.append({
            "output": f"pl_tile{idx:02d}subtile{j:02d}.osm.pbf",
            "output_format": "pbf",
            "bbox": sub_bbox,
        })

SUBTILE_BATCH_SIZE = 2
subtile_config_paths = []
for batch_idx in range(0, len(subtile_extracts), SUBTILE_BATCH_SIZE):
    batch = subtile_extracts[batch_idx:batch_idx + SUBTILE_BATCH_SIZE]
    config = {"directory": str(EXTRACT_DIR), "extracts": batch}
    config_path = EXTRACT_DIR / f"pl_subtile_extract_config_batch{batch_idx // SUBTILE_BATCH_SIZE:02d}.json"
    config_path.write_text(json.dumps(config, indent=2))
    subtile_config_paths.append(config_path)

print(f"{len(subtile_extracts)} subtile, {len(subtile_config_paths)} köteg")

In [ ]:
import subprocess

for i, config_path in enumerate(subtile_config_paths):
    cmd = [
        "osmium", "extract",
        "--config", str(config_path),
        "--strategy", "complete_ways",
        "--overwrite",
        "--progress",
        str(OSM_DIR / "pl-latest.osm.pbf"),
    ]
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"köteg {i+1}/{len(subtile_config_paths)}: {time.time()-t0:.1f} mp, returncode={result.returncode}")
    if result.returncode != 0:
        print(result.stderr[-1000:])
        break

In [ ]:
expected_subtiles = [e["output"] for e in subtile_extracts]
print(f"Várt subtile-fájlok: {len(expected_subtiles)}")

all_ok = True
for name in expected_subtiles:
    p = EXTRACT_DIR / name
    if not p.exists():
        print(f"  [HIÁNYZIK] {name}")
        all_ok = False
    elif p.stat().st_size < 1000:
        print(f"  [GYANÚS] {name} ({p.stat().st_size} byte)")
        all_ok = False
    else:
        print(f"  [OK] {name} ({p.stat().st_size:,} byte)")

print(f"\nMinden subtile rendben: {all_ok}")

In [ ]:
process_country_tiles_streaming("hu")

In [ ]:
process_country_tiles_streaming("sk")

In [ ]:
process_country_tiles_streaming("cz")

In [ ]:
process_country_tiles_streaming("pl")